In [1]:
import polars as pl 
from pathlib import Path 
import logging
import json
import time
import requests

logger = logging.getLogger(__name__)

STEP_X3_OUTPUT_PATH = Path("/cluster/project/beltrao/kdammer/master_thesis/data/Pipeline/t11_RM_TM_updated_CF_pipeline/cf_pdb_structure_similarity/aggregate_cf_for_pdb_eval_with_combfold_results_paths.parquet") 
REFERENCE_PDB_DIR = Path("/cluster/project/beltrao/kdammer/master_thesis/data/reference_pdb")

In [2]:
df_stepX3 = pl.read_parquet(STEP_X3_OUTPUT_PATH)

In [3]:
print(df_stepX3.columns)

['complex_ac', 'identifiers', 'pred_1', 'pred_2', 'pred_3', 'correct_pred_count', 'correct_pred_rank', 'CF_1_n_assemblies', 'CF_1_confidence', 'CF_1_reason', 'CF_2_n_assemblies', 'CF_2_confidence', 'CF_2_reason', 'CF_3_n_assemblies', 'CF_3_confidence', 'CF_3_reason', 'CF_true_n_assemblies', 'CF_true_confidence', 'CF_true_reason', 'combfold_job_ids', 'n_pairs_used', 'pairs_used', 'n_proteins', 'pdb_id', 'match_class', 'pdb_stoichiometry', 'CP_stochiometry', 'reference_pdb_model', 'found_match_reference_pdb_model', 'Combfold_result_path', 'n_combfold_outputs']


In [4]:
assert df_stepX3["found_match_reference_pdb_model"].all(), "Not all rows have a matching reference PDB model. Please check the data. This should have been filtered in the previous step X3."
df_relevant_cols = df_stepX3[["complex_ac", "identifiers", "pdb_id", "reference_pdb_model", "Combfold_result_path", "n_pairs_used", "pairs_used", "correct_pred_rank", "n_combfold_outputs"]]

In [5]:
def _get_reference_pdb_path(pdb_id: str, reference_pdb_model: str, reference_pdb_dir: Path = REFERENCE_PDB_DIR) -> Path:
    """
    Given a PDB ID and a reference PDB model, return the path to the corresponding reference PDB file.
    """
    pdb_id = pdb_id.lower()
    if reference_pdb_model == "au":
        file_path_without_suffix =  (reference_pdb_dir / pdb_id / pdb_id)
    else:
        file_path_without_suffix = (reference_pdb_dir / pdb_id / f"{pdb_id}-assembly{reference_pdb_model}")

    if file_path_without_suffix.with_suffix(".pdb").exists():
        return file_path_without_suffix.with_suffix(".pdb")
    elif file_path_without_suffix.with_suffix(".cif").exists():
        return file_path_without_suffix.with_suffix(".cif")
    raise FileNotFoundError(f"Neither .pdb nor .cif file found for {pdb_id} at {file_path_without_suffix}.")

# resolve reference pdb path per row 
df_relevant_cols = df_relevant_cols.with_columns(
    pl.struct(["pdb_id", "reference_pdb_model"])
    .map_elements(
        lambda row: str(_get_reference_pdb_path(row["pdb_id"], row["reference_pdb_model"])),
        return_dtype=pl.Utf8,
    )
    .alias("reference_pdb_path")
)

In [6]:
def _get_combfold_output_paths(combfold_result_path: str, n_combfold_outputs: int) -> list[Path]:
    """
    Given a Combfold_result_path, list the assembled CombFold output files under
    `{combfold_result_path}/assembled_results/`, sorted for a stable/reproducible order.

    Raises if the number of files found does not match `n_combfold_outputs` -- we don't want
    to silently proceed with a mismatched count.
    """
    combfold_result_path = Path(combfold_result_path)
    if n_combfold_outputs == 0:
        return []

    assembled_results_dir = combfold_result_path / "assembled_results"

    if not assembled_results_dir.is_dir():
        raise FileNotFoundError(f"assembled_results dir does not exist: {assembled_results_dir}")

    assert all(p.suffix in (".txt", ".pdb") for p in assembled_results_dir.iterdir()), f"Unexpected file extension found in {assembled_results_dir}. Expected only .txt or .pdb files."
    output_paths = sorted(assembled_results_dir.glob("*.pdb"))

    if len(output_paths) != n_combfold_outputs:
        raise ValueError(
            f"Expected {n_combfold_outputs} CombFold outputs in {assembled_results_dir}, "
            f"found {len(output_paths)}."
        )

    return output_paths

# %%
# resolve the list of CombFold output paths per row, then explode into one row per output
df_relevant_cols = df_relevant_cols.with_columns(
    pl.struct(["Combfold_result_path", "n_combfold_outputs"])
    .map_elements(
        lambda row: [
            str(p)
            for p in _get_combfold_output_paths(row["Combfold_result_path"], row["n_combfold_outputs"])
        ],
        return_dtype=pl.List(pl.Utf8),
    )
    .alias("combfold_output_path")
)

df_long = df_relevant_cols.explode("combfold_output_path")

# sanity checks: row count should scale by n_combfold_outputs (treating 0 -> 1 row from explode),
# and only rows with n_combfold_outputs == 0 should have a null combfold_output_path
expected_n_rows = (
    df_relevant_cols["n_combfold_outputs"].clip(lower_bound=1).sum()
)
assert df_long.height == expected_n_rows, (
    f"Expected {expected_n_rows} rows after exploding, got {df_long.height}."
)

expected_null_outputs = (df_relevant_cols["n_combfold_outputs"] == 0).sum()
actual_null_outputs = df_long["combfold_output_path"].null_count()
assert actual_null_outputs == expected_null_outputs, (
    f"Expected {expected_null_outputs} null combfold_output_path rows (n_combfold_outputs == 0), "
    f"got {actual_null_outputs}."
)

assert df_long["reference_pdb_path"].null_count() == 0, "Found null reference_pdb_path."

logger.info(f"Built long df with {df_long.height} rows from {df_relevant_cols.height} complexes.")

# %%
df_long

complex_ac,identifiers,pdb_id,reference_pdb_model,Combfold_result_path,n_pairs_used,pairs_used,correct_pred_rank,n_combfold_outputs,reference_pdb_path,combfold_output_path
str,str,str,str,str,i64,str,str,i64,str,str
"""CPX-940""","""P25604(1)|P42939(1)|Q02767(1)|…","""2P22""","""1""","""/cluster/project/beltrao/kdamm…",6,"""['P25604_P25604', 'P25604_P429…","""1""",0,"""/cluster/project/beltrao/kdamm…",null
"""CPX-1320""","""CHEBI:29105(1)|P32628(1)|Q0289…","""1X3Z""","""1""","""/cluster/project/beltrao/kdamm…",3,"""['P32628_P32628', 'P32628_Q028…","""1""",3,"""/cluster/project/beltrao/kdamm…","""/cluster/project/beltrao/kdamm…"
"""CPX-1320""","""CHEBI:29105(1)|P32628(1)|Q0289…","""1X3Z""","""1""","""/cluster/project/beltrao/kdamm…",3,"""['P32628_P32628', 'P32628_Q028…","""1""",3,"""/cluster/project/beltrao/kdamm…","""/cluster/project/beltrao/kdamm…"
"""CPX-1320""","""CHEBI:29105(1)|P32628(1)|Q0289…","""1X3Z""","""1""","""/cluster/project/beltrao/kdamm…",3,"""['P32628_P32628', 'P32628_Q028…","""1""",3,"""/cluster/project/beltrao/kdamm…","""/cluster/project/beltrao/kdamm…"
"""CPX-1640""","""P14736(1)|P32628(1)""","""2QSF""","""1""","""/cluster/project/beltrao/kdamm…",3,"""['P14736_P14736', 'P14736_P326…","""1""",3,"""/cluster/project/beltrao/kdamm…","""/cluster/project/beltrao/kdamm…"
…,…,…,…,…,…,…,…,…,…,…
"""CPX-1677""","""P22219(1)|P22543(1)|Q02948(1)|…","""5DFZ""","""1""","""/cluster/project/beltrao/kdamm…",8,"""['P22219_P22219', 'P22219_P225…","""1""",4,"""/cluster/project/beltrao/kdamm…","""/cluster/project/beltrao/kdamm…"
"""CPX-1706""","""P28496(1)|P38703(1)|Q03579(2)""","""8QTN""","""1""","""/cluster/project/beltrao/kdamm…",6,"""['P28496_P28496', 'P28496_P387…","""none""",3,"""/cluster/project/beltrao/kdamm…","""/cluster/project/beltrao/kdamm…"
"""CPX-1706""","""P28496(1)|P38703(1)|Q03579(2)""","""8QTN""","""1""","""/cluster/project/beltrao/kdamm…",6,"""['P28496_P28496', 'P28496_P387…","""none""",3,"""/cluster/project/beltrao/kdamm…","""/cluster/project/beltrao/kdamm…"


In [19]:
df_long = df_long[9:10]

In [8]:
subset

complex_ac,identifiers,pdb_id,reference_pdb_model,Combfold_result_path,n_pairs_used,pairs_used,correct_pred_rank,n_combfold_outputs,reference_pdb_path,combfold_output_path
str,str,str,str,str,i64,str,str,i64,str,str
"""CPX-1677""","""P22219(1)|P22543(1)|Q02948(1)|…","""5DFZ""","""1""","""/cluster/project/beltrao/kdamm…",8,"""['P22219_P22219', 'P22219_P225…","""1""",4,"""/cluster/project/beltrao/kdamm…","""/cluster/project/beltrao/kdamm…"


In [17]:
# get sifts

SIFTS_FILENAME_TEMPLATE = "{pdb_id}_sifts.json"
SIFTS_URL_TEMPLATE = "https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id}"


def _sifts_path(pdb_id: str) -> Path:
    return REFERENCE_PDB_DIR / pdb_id / SIFTS_FILENAME_TEMPLATE.format(pdb_id=pdb_id)


def _download_sifts(pdb_id: str, timeout: int = 15) -> dict | None:
    url = SIFTS_URL_TEMPLATE.format(pdb_id=pdb_id)
    try:
        resp = requests.get(url, timeout=timeout)
    except requests.RequestException as exc:
        logger.warning(f"{pdb_id}: SIFTS request failed ({exc}) — skipping")
        return None

    if resp.status_code == 404:
        logger.warning(f"{pdb_id}: SIFTS API returned 404 (no UniProt mapping) — skipping")
        return None
    if resp.status_code != 200:
        logger.warning(f"{pdb_id}: SIFTS API returned status {resp.status_code} — skipping")
        return None

    try:
        return resp.json()
    except ValueError:
        logger.warning(f"{pdb_id}: SIFTS response was not valid JSON — skipping")
        return None


def ensure_sifts_files(df, pdb_id_col: str = "pdb_id", sleep_between_requests: float = 0.2):
    """For every unique pdb_id in df, ensure a cached SIFTS UniProt-mapping file
    exists under REFERENCE_PDB_DIR/{pdb_id}/. Downloads if missing.

    Returns dict: pdb_id -> bool (True if file present/available after this call).
    """
    unique_pdb_ids = sorted({str(pid).lower() for pid in df[pdb_id_col].unique()})
    logger.info(f"Checking SIFTS UniProt mappings for {len(unique_pdb_ids)} unique PDB IDs")

    status = {}
    for pdb_id in unique_pdb_ids:
        pdb_dir = REFERENCE_PDB_DIR / pdb_id
        if not pdb_dir.is_dir():
            logger.warning(f"{pdb_id}: no reference directory {pdb_dir} — skipping SIFTS check")
            status[pdb_id] = False
            continue

        sifts_path = _sifts_path(pdb_id)
        if sifts_path.exists():
            logger.debug(f"{pdb_id}: SIFTS file already present at {sifts_path}")
            status[pdb_id] = True
            continue

        logger.info(f"{pdb_id}: SIFTS file missing — downloading")
        data = _download_sifts(pdb_id)
        if data is None:
            status[pdb_id] = False
            continue

        sifts_path.write_text(json.dumps(data))
        logger.info(f"{pdb_id}: SIFTS file written to {sifts_path}")
        status[pdb_id] = True

        time.sleep(sleep_between_requests)  # be polite to the EBI API

    n_ok = sum(status.values())
    n_missing = len(status) - n_ok
    if n_missing:
        logger.warning(f"SIFTS mapping unavailable for {n_missing}/{len(status)} PDB IDs: "
                        f"{[k for k, v in status.items() if not v]}")

    return status

In [20]:
#download sifsts, if it doent exist already
ensure_sifts_files(df_long)

{'5dfz': True}

In [21]:
import json, re

def _build_chain_mapping(reference_pdb_path: str, combfold_output_path: str) -> dict[str, tuple[str, str]]:
    pdb_dir = Path(reference_pdb_path).parent
    sifts = json.loads((pdb_dir / SIFTS_FILENAME_TEMPLATE.format(pdb_id=pdb_dir.name)).read_text())
    pdb_chains = {unp: e["mappings"][0]["chain_id"] for unp, e in sifts[pdb_dir.name]["UniProt"].items()}
    assert all(len(e["mappings"]) == 1 for e in sifts[pdb_dir.name]["UniProt"].values()), "expected 1 mapping per uniprot (heteromer)"

    chain_list = Path(combfold_output_path).parent.parent / "_unified_representation" / "assembly_output" / "chain.list"
    cf_chains = dict(re.match(r"^(\w+)_(\w+)\.pdb$", line).groups() for line in chain_list.read_text().split())

    assert set(cf_chains) == set(pdb_chains), f"uniprot mismatch: {set(cf_chains) ^ set(pdb_chains)}"
    return {unp: (cf_chains[unp], pdb_chains[unp]) for unp in cf_chains}


df_long = df_long.with_columns(
    pl.struct(["reference_pdb_path", "combfold_output_path"])
    .map_elements(lambda r: _build_chain_mapping(r["reference_pdb_path"], r["combfold_output_path"]), return_dtype=pl.Object)
    .alias("chain_mapping")
)

In [23]:
from Bio.PDB import PDBParser, MMCIFParser, Superimposer
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1
import numpy as np

def _get_chain_ca(struct_path: str, chain_id: str):
    """Returns (sequence_str, list[CA Atom]) for one chain, standard amino acids only."""
    parser = MMCIFParser(QUIET=True) if struct_path.endswith(".cif") else PDBParser(QUIET=True)
    structure = parser.get_structure("x", struct_path)
    chain = structure[0][chain_id]

    residues = [r for r in chain if is_aa(r, standard=True) and "CA" in r]
    seq = "".join(seq1(r.get_resname()) for r in residues)
    ca_atoms = [r["CA"] for r in residues]
    return seq, ca_atoms


def _aligned_ca_pairs(seq_cf, ca_cf, seq_pdb, ca_pdb):
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")  
    aligner.open_gap_score = -10                                          
    aligner.extend_gap_score = -0.5                                       
    alignment = aligner.align(seq_cf, seq_pdb)[0]
    aligned_cf, aligned_pdb = alignment.indices

    matched_cf, matched_pdb, n_identical = [], [], 0
    for i_cf, i_pdb in zip(aligned_cf, aligned_pdb):
        if i_cf == -1 or i_pdb == -1:
            continue
        matched_cf.append(ca_cf[i_cf])
        matched_pdb.append(ca_pdb[i_pdb])
        n_identical += seq_cf[i_cf] == seq_pdb[i_pdb]

    identity = n_identical / len(matched_cf) if matched_cf else 0.0   # also guards the divide-by-zero
    return matched_cf, matched_pdb, identity

def compute_rmsd(combfold_output_path: str, reference_pdb_path: str, chain_mapping: dict[str, tuple[str, str]]):
    """chain_mapping: {uniprot: (cf_chain_id, pdb_chain_id)}. Returns (rmsd, min_identity)."""
    all_cf_atoms, all_pdb_atoms, identities = [], [], []

    for uniprot, (cf_chain, pdb_chain) in chain_mapping.items():
        seq_cf, ca_cf = _get_chain_ca(combfold_output_path, cf_chain)
        seq_pdb, ca_pdb = _get_chain_ca(reference_pdb_path, pdb_chain)
        assert seq_cf and seq_pdb, f"{uniprot}: empty sequence (cf={len(seq_cf)}, pdb={len(seq_pdb)})"

        matched_cf, matched_pdb, identity = _aligned_ca_pairs(seq_cf, ca_cf, seq_pdb, ca_pdb)
        assert matched_cf, f"{uniprot}: no aligned residues between CF chain {cf_chain} and PDB chain {pdb_chain}"

        all_cf_atoms += matched_cf
        all_pdb_atoms += matched_pdb
        identities.append(identity)

    sup = Superimposer()
    sup.set_atoms(all_pdb_atoms, all_cf_atoms)  # fixed=pdb, moving=cf
    return sup.rms, min(identities)